In [ ]:
import pandas as pd
from openai import OpenAI
import json

client = OpenAI()
 
# Load the knowledge graphs
with open("404KG.txt", "r", encoding="utf-8") as f: #Update with the name of the TXT with the KG
    graphs_text = f.read() 
 
prompt = f"""

    You are provided with a set of knowledge graphs extracted from sentences from the scientific literature, all of which contain the term *ancest*. 

    Your tasks are: 

        Identify conceptual clusters/categories that summarize the implicit definitions or uses of "ancestry" found in the knowledge graphs

        Name each cluster with a concise, descriptive label

        Provide a short definition of each cluster (1-3 sentences)

        Explain the reasoning behind the clustering using only evidence visible in the knowledge graphs

    Important instructions:

        Clusters must emerge from the nodes and the relationships in the knowledge graphs; do not use external theories 

        Base all reasoning strictly on the content and structure of the provided knowledge graphs. 

        Clusters must be mutually exclusive, without overlapping definition and description. 
  
    Output format:  

        Return a JSON array in which each element represents one conceptual cluster, with the following fields:

        {{
            "label": "<cluster/category label>",
            
            "definition": "<1–3 sentence cluster definition>",
            
            "reasoning": "<1–2 sentence explanation of why this cluster emerges from the knowledge graphs>"
        }}

        Knowledge graphs:
            {graphs_text}"""

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

raw_output = completion.choices[0].message.content.strip()

if raw_output.startswith("```"):
    raw_output = raw_output.strip("`")
    raw_output = raw_output.replace("json\n", "", 1).strip()

try:
    clusters = json.loads(raw_output)
except json.JSONDecodeError:
    clusters = [{"label": "Error", "definition": "", "reasoning": raw_output}]

with open("clusters.json", "w", encoding="utf-8") as f:
    json.dump(clusters, f, ensure_ascii=False, indent=2)

print("Clustering complete. Output saved to clusters.json.")


Clustering complete. Output saved to clusters.json.
